<a href="https://colab.research.google.com/github/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_04_QC_M34099_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 Laboratory: Multi-Qubit Circuits and Entanglement

[![View in GitHub](https://img.shields.io/badge/View%20in-GitHub-black?logo=github)](https://github.com/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_04_QC_M34099_Student.ipynb)

**M34099 Quantum Computing: Level 7 (2026/27)**

**Problem:** A supplier reports perfect 00/11 matching in the Z basis and claims that this proves entanglement.

You will build two-qubit circuits, examine CNOT, prepare a Bell state, compare Z-basis and X-basis measurements, and run the same Bell circuit on a simulator or an IBM quantum computer. Run the cells in order.

> **Saving your work:** In Colab, select **File**, then **Save a copy in Drive** before editing. Never place an API key or CRN in a visible notebook cell.


## 1. Install and import the required packages

Run the installation cell once. If Google Colab requests a restart, select **Runtime > Restart session**, then continue from the imports cell.


In [ ]:
# Install Qiskit, Aer, IBM Runtime and circuit drawing support.
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc


In [ ]:
# Import NumPy for vector and matrix calculations.
import numpy as np

# Import the Qiskit circuit constructor and circuit translator.
from qiskit import QuantumCircuit, transpile

# Import exact statevector, density-matrix and partial-trace tools.
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace

# Import the local ideal quantum simulator.
from qiskit_aer import AerSimulator

# Import the histogram plotting function.
from qiskit.visualization import plot_histogram

# Set the number of repeated measurements.
SHOTS = 4096

# Set a fixed seed for reproducible simulator results.
SEED = 34099

# Create the local simulator.
simulator = AerSimulator()

# Confirm that the setup is complete.
print("Setup complete.")


## 2. Two-qubit states and circuit order

A two-qubit state has four computational basis states:

$$|00\rangle,\ |01\rangle,\ |10\rangle,\ |11\rangle.$$

Qiskit displays basis labels in the order $|q_1q_0\rangle$. Circuit gates act from left to right in time.


## 3. Task 1: Form a two-qubit product state

The tensor product combines separate qubit states. For $|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}$, the state $|0\rangle\otimes|+\rangle$ has equal amplitudes for $|00\rangle$ and $|01\rangle$.


In [ ]:
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)  # Create the one-qubit plus state.
zero = np.array([1, 0], dtype=complex)  # Create the one-qubit zero state.
product_state = np.kron(zero, plus)  # Combine q1 in zero with q0 in plus.
probabilities = np.abs(product_state) ** 2  # Convert amplitudes into probabilities.
print("Statevector:", product_state)  # Display the four amplitudes.
print("Probabilities:", probabilities)  # Display the four measurement probabilities.


## 4. Task 2: Observe the CNOT gate

CNOT uses qubit 0 as the control and qubit 1 as the target. The target changes only when the control is 1.


In [ ]:
def cnot_output(control, target):  # Define a reusable CNOT demonstration.
    circuit = QuantumCircuit(2)  # Create a circuit containing two qubits.
    if control == 1:  # Check whether control qubit q0 must start in state one.
        circuit.x(0)  # Change control qubit q0 from zero to one.
    if target == 1:  # Check whether target qubit q1 must start in state one.
        circuit.x(1)  # Change target qubit q1 from zero to one.
    circuit.cx(0, 1)  # Apply CNOT with q0 as control and q1 as target.
    return Statevector.from_instruction(circuit)  # Return the exact final state.
for control_bit in [0, 1]:  # Use both possible control-bit values.
    for target_bit in [0, 1]:  # Use both possible target-bit values.
        output_state = cnot_output(control_bit, target_bit)  # Calculate the CNOT output.
        print(f"Input q0={control_bit}, q1={target_bit}:", output_state.probabilities_dict())  # Display the output probabilities.


## 5. Task 3: Build a Bell-state circuit

The circuit begins in $|00\rangle$.

1. Hadamard places qubit 0 into an equal superposition.
2. CNOT links qubit 1 to qubit 0.
3. The final state is $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt{2}$.


In [ ]:
bell_circuit = QuantumCircuit(2)  # Create a circuit with two qubits in state zero.
bell_circuit.h(0)  # Place qubit 0 into an equal superposition.
bell_circuit.cx(0, 1)  # Entangle qubit 1 with qubit 0.
bell_state = Statevector.from_instruction(bell_circuit)  # Calculate the exact two-qubit state.
reduced_q0 = partial_trace(bell_state, [1])  # Trace out q1 to inspect q0 alone.
print("Bell state:", bell_state.data)  # Display the Bell-state amplitudes.
print("Basis probabilities:", bell_state.probabilities_dict())  # Display the ideal basis probabilities.
print("Reduced state of q0:\n", reduced_q0.data)  # Show that q0 alone has equal zero and one probabilities.
display(bell_circuit.draw("mpl"))  # Draw the Bell-state circuit.


## 6. Task 4: Measure in the Z and X bases

Qiskit measures in the Z basis by default. Applying Hadamard to both qubits before measurement changes an X-basis measurement into a Z-basis measurement.

For the Bell state, the two results match in both bases. Small differences between the `00` and `11` counts result from finite sampling.


In [ ]:
def build_bell_measurement(basis="Z"):  # Define a reusable measured Bell circuit.
    circuit = bell_circuit.copy()  # Copy the Bell circuit built in the previous section.
    if basis.upper() == "X":  # Check whether an X-basis measurement is required.
        circuit.h([0, 1])  # Rotate both qubits from the X basis to the Z basis.
    circuit.measure_all()  # Measure both qubits into a new classical register named meas.
    return circuit  # Return the completed measurement circuit.
def run_on_simulator(circuit):  # Define a reusable simulator function.
    compiled_circuit = transpile(circuit, simulator, seed_transpiler=SEED)  # Translate the circuit for Aer.
    result = simulator.run(compiled_circuit, shots=SHOTS, seed_simulator=SEED).result()  # Run repeated measurements.
    return result.get_counts()  # Return the measured bit-string counts.
def matching_frequency(counts):  # Define the fraction of matching outcomes.
    matching_counts = counts.get("00", 0) + counts.get("11", 0)  # Add the 00 and 11 results.
    return matching_counts / sum(counts.values())  # Divide matching results by all results.
bell_z_circuit = build_bell_measurement("Z")  # Build the Z-basis Bell experiment.
bell_x_circuit = build_bell_measurement("X")  # Build the X-basis Bell experiment.
z_counts = run_on_simulator(bell_z_circuit)  # Run the Z-basis experiment.
x_counts = run_on_simulator(bell_x_circuit)  # Run the X-basis experiment.
print("Z-basis counts:", z_counts)  # Display the Z-basis counts.
print("Z matching frequency:", matching_frequency(z_counts))  # Display the Z matching frequency.
print("X-basis counts:", x_counts)  # Display the X-basis counts.
print("X matching frequency:", matching_frequency(x_counts))  # Display the X matching frequency.
display(bell_z_circuit.draw("mpl"))  # Draw the measured Z-basis circuit.
display(bell_x_circuit.draw("mpl"))  # Draw the measured X-basis circuit.
display(plot_histogram([z_counts, x_counts], legend=["Bell Z", "Bell X"]))  # Compare both count distributions.


## 7. Task 5: Compare entanglement with a classical mixture

A classical 00/11 mixture produces the same perfect matching in the Z basis. It is not entangled because it contains no coherence between $|00\rangle$ and $|11\rangle$.

The Bell density matrix has nonzero off-diagonal terms. The mixture does not. In the X basis, the Bell state gives matching results every time, while the classical mixture gives matching results about half of the time.


In [ ]:
rho_bell = DensityMatrix(bell_state)  # Convert the Bell state into a density matrix.
ket00 = Statevector.from_label("00")  # Create the two-qubit state 00.
ket11 = Statevector.from_label("11")  # Create the two-qubit state 11.
rho_00 = DensityMatrix(ket00).data  # Convert state 00 into a density-matrix array.
rho_11 = DensityMatrix(ket11).data  # Convert state 11 into a density-matrix array.
rho_mixture = 0.5 * rho_00 + 0.5 * rho_11  # Form an equal classical mixture of 00 and 11.
print("Bell density matrix:\n", np.round(rho_bell.data, 2))  # Display the Bell density matrix.
print("Classical mixture:\n", np.round(rho_mixture, 2))  # Display the classical-mixture density matrix.


## 8. Optional run on an IBM quantum computer

This section sends the same `bell_z_circuit` used by the simulator to real IBM quantum hardware. Complete it only when your tutor confirms that IBM Quantum Classroom access is available.

- Keep `RUN_ON_REAL_HARDWARE = False` until access is ready.
- Use your own IBM Cloud API key and the classroom CRN supplied through Moodle.
- Hidden input prevents credentials from appearing on screen.
- Never save credentials in the notebook, Google Drive, Moodle or GitHub.

Real hardware is noisy, so a small number of `01` and `10` results may appear.


In [ ]:
RUN_ON_REAL_HARDWARE = True  # Change this value to True only when IBM access is ready.
print("Real-hardware execution enabled:", RUN_ON_REAL_HARDWARE)  # Display the current safety setting.


In [ ]:
if RUN_ON_REAL_HARDWARE:  # Continue only when real-hardware execution is enabled.
    from getpass import getpass  # Provide hidden text input for credentials.
    from qiskit_ibm_runtime import QiskitRuntimeService  # Provide access to IBM Quantum services.
    ibm_api_key = getpass("Enter your personal IBM Cloud API key: ")  # Read the API key without displaying it.
    classroom_crn = getpass("Enter the classroom instance CRN: ")  # Read the classroom CRN without displaying it.
    ibm_service = QiskitRuntimeService(channel="ibm_quantum_platform", token=ibm_api_key, instance=classroom_crn)  # Connect to the classroom service.
    del ibm_api_key, classroom_crn  # Remove the plaintext credential variables.
    print("Connected to IBM Quantum.")  # Confirm that the connection is ready.
else:  # Handle the safe default setting.
    print("Hardware connection skipped.")  # Confirm that no credentials were requested.


In [ ]:
if RUN_ON_REAL_HARDWARE:  # Continue only when the IBM service is connected.
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager  # Provide device-aware circuit translation.
    from qiskit_ibm_runtime import SamplerV2  # Provide the IBM Runtime sampling tool.
    HARDWARE_SHOTS = 1024  # Set a modest number of real-device measurements.
    backend = ibm_service.least_busy(operational=True, simulator=False, min_num_qubits=2)  # Select an available real device.
    pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=1)  # Create a translator for the device.
    hardware_circuit = pass_manager.run(bell_z_circuit)  # Translate the same Bell circuit used by the simulator.
    sampler = SamplerV2(mode=backend)  # Connect the sampler to the selected device.
    hardware_job = sampler.run([hardware_circuit], shots=HARDWARE_SHOTS)  # Submit the circuit to the device queue.
    print("Backend:", backend.name)  # Display the selected device name.
    print("Job ID:", hardware_job.job_id())  # Display the job identifier.
    hardware_result = hardware_job.result()  # Wait for the device to complete the job.
    hardware_counts = hardware_result[0].data.meas.get_counts()  # Extract counts from the meas register.
    print("Hardware counts:", hardware_counts)  # Display the real-device counts.
    print("Hardware matching frequency:", matching_frequency(hardware_counts))  # Display the real-device matching frequency.
    display(plot_histogram(hardware_counts, title=f"IBM hardware: {backend.name}"))  # Plot the real-device results.
else:  # Handle the safe default setting.
    print("No hardware job submitted.")  # Confirm that the hardware section was skipped.


## Key result

Perfect Z-basis matching alone does not prove entanglement because a classical 00/11 mixture gives the same Z-basis pattern. Matching measurements in the X basis provide evidence of the coherence expected from the Bell state.

## References

- [IBM Quantum Learning: Multiple systems](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/multiple-systems)
- [Qiskit Statevector API](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.Statevector)
- [Qiskit DensityMatrix API](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.DensityMatrix)
- [IBM Quantum: Run jobs with Qiskit Runtime](https://quantum.cloud.ibm.com/docs/guides/run-jobs)
- [Qiskit Aer documentation](https://qiskit.github.io/qiskit-aer/)

Accessed 12 October 2026.
